# Exact sci-mtChIL-seq reanalysis used for final Fig. 2f

The analysis cells are copied from the final figure-generating notebook. Only input and output paths are changed.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "analyses").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the Gao2026 code root")

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
GAO2026_DATA_ROOT = Path(
    os.environ.get("GAO2026_DATA_ROOT", REPO_ROOT / "data")
).expanduser().resolve()
GAO2026_OUTPUT_ROOT = Path(
    os.environ.get("GAO2026_OUTPUT_ROOT", REPO_ROOT / "outputs")
).expanduser().resolve()


# seqFISH/ChIL: minimal notebook for two final figures

This notebook is trimmed to only the two figures used in the paper:

1) Active/Inactive CPT ratio by non-overlapping ChIL rank bins.
2) nCell_posi distributions by pos_ratio bins within each target.

## Required files

- avgCPT_rnapU1D5_pairU5D5.csv

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.grid'] = True

In [ ]:
# ---- Path settings (modify as needed) ----
path_u5d5 = GAO2026_DATA_ROOT / "05_sci_mtChIL_seq/avgCPT_rnapU1D5_pairU5D5.csv"

# ---- Load data ----
u5d5 = pd.read_csv(path_u5d5)

# Ensure gene is string
u5d5['gene'] = u5d5['gene'].astype(str)

print('Loaded: u5d5=%s' % (len(u5d5)))

In [ ]:
def add_posi_nega_norm(df):
    """Add posi_norm / nega_norm from avg_posi and avg_nega."""
    df = df.copy()
    s = (df['avg_posi'].astype(float) + df['avg_nega'].astype(float))
    df['posi_norm'] = (df['avg_posi'].astype(float) / s).fillna(0.0)
    df['nega_norm'] = 1.0 - df['posi_norm']
    return df

def add_pos_ratio(df):
    """Add pos_ratio = nCell_posi/(nCell_posi+nCell_nega)."""
    df = df.copy()
    s = (df['nCell_posi'].astype(float) + df['nCell_nega'].astype(float))
    df['pos_ratio'] = (df['nCell_posi'].astype(float) / s).fillna(0.0)
    return df

In [ ]:
# Add normalized columns and pos_ratio
u5d5_n = add_posi_nega_norm(add_pos_ratio(u5d5))

# Ranking scores (ChIL)
chil_rank = (
    u5d5_n[['gene', 'pos_ratio']]
    .groupby('gene', as_index=False)['pos_ratio']
    .mean()
    .rename(columns={'pos_ratio': 'chil_pos_ratio'})
)

print('Genes available for ChIL ranking:', chil_rank['gene'].nunique())

In [ ]:
# Select dataset for figure generation
ds_name = 'pairU5D5'
dsn = u5d5_n

## Non-overlapping ChIL-rank bins (default 500 genes per bin)

Genes are ranked by `chil_pos_ratio` and split into non-overlapping bins.
Each bin is tested for deviation of $\log_2(\mathrm{ratio})$ from 0 using one-sample Wilcoxon
and BH-FDR correction across all bins and targets.

In [ ]:
from matplotlib.patches import Patch
from scipy.stats import wilcoxon

# ----------------------------
# Settings (adjust as needed)
# ----------------------------
DS_NAME = globals().get("ds_name", "pairU5D5")
dsn_local = globals().get("dsn", None)
chil_rank_local = globals().get("chil_rank", None)

TARGETS = ["H3K27ac", "H3K4me3", "H3K27me3"]
RANK_SCORE_COL = "chil_pos_ratio"
EPS_RATIO = 1e-2

# Configurable bin size (default: 500)
BIN_SIZE = int(globals().get("BIN_SIZE_RANK", 500))
MAX_RANK_TO_PLOT = None  # set to an int to limit bins; None uses all

# Plot appearance
FONT_FAMILY = "Arial"
FIG_W, FIG_H = 9.5, 4.5
LABEL_FONTSIZE = 16
TICK_FONTSIZE = 13
LEGEND_FONTSIZE = 12

Y_SCALE = "log"
SHOW_POINTS = False
POINT_ALPHA = 0.08
POINT_SIZE = 6

TARGET_GAP = 1.0
GROUP_SPAN = 0.72
VIOLIN_WIDTH = 0.11
BOX_WIDTH = 0.06

SHOW_NS = True
STAR_FONTSIZE = 12
STAR_Y_MULT = 1.12

LEGEND_OUTSIDE_RIGHT = True
LEGEND_RIGHT_MARGIN = 0.78

OUT_DIR = GAO2026_OUTPUT_ROOT / "05_sci_mtChIL_seq" / f"ratio_violin_boxplot_{DS_NAME}_chilRank_bins{BIN_SIZE}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PDF = OUT_DIR / f"violin_boxplot_ratio_{DS_NAME}_bins{BIN_SIZE}.pdf"

# ----------------------------
# Safety checks
# ----------------------------
if dsn_local is None:
    raise RuntimeError("dsn is not defined. Run the dataset selection cell first.")
if chil_rank_local is None:
    raise RuntimeError("chil_rank is not defined. Run the ranking cell first.")
for c in ["gene", "target", "avg_posi", "avg_nega"]:
    if c not in dsn_local.columns:
        raise RuntimeError(f"dsn is missing column '{c}'. columns={list(dsn_local.columns)}")
for c in ["gene", RANK_SCORE_COL]:
    if c not in chil_rank_local.columns:
        raise RuntimeError(f"chil_rank is missing column '{c}'. columns={list(chil_rank_local.columns)}")

# ----------------------------
# Helpers (reuse if already defined, else define)
# ----------------------------
if "bh_fdr" in globals():
    bh_fdr = globals()["bh_fdr"]
else:
    def bh_fdr(pvals: np.ndarray) -> np.ndarray:
        pvals = np.asarray(pvals, dtype=float)
        qvals = np.full_like(pvals, np.nan, dtype=float)
        msk = np.isfinite(pvals)
        if not np.any(msk):
            return qvals
        p = pvals[msk]
        m = p.size
        order = np.argsort(p)
        ranked = p[order]
        ranks = np.arange(1, m + 1, dtype=float)
        q = ranked * m / ranks
        q = np.minimum.accumulate(q[::-1])[::-1]
        q = np.clip(q, 0.0, 1.0)
        q_back = np.empty_like(q)
        q_back[order] = q
        qvals[msk] = q_back
        return qvals

if "stars_from_q" in globals():
    stars_from_q = globals()["stars_from_q"]
else:
    def stars_from_q(q: float) -> str:
        if not np.isfinite(q):
            return "NA"
        if q <= 1e-4:
            return "****"
        if q <= 1e-3:
            return "***"
        if q <= 1e-2:
            return "**"
        if q <= 5e-2:
            return "*"
        return "NS"

# ----------------------------
# Build ranked genes + assign non-overlapping bins
# ----------------------------
rank_tbl = chil_rank_local[["gene", RANK_SCORE_COL]].dropna().copy()
rank_tbl["gene"] = rank_tbl["gene"].astype(str)
rank_tbl = rank_tbl.groupby("gene", as_index=False)[RANK_SCORE_COL].max()
rank_tbl = rank_tbl.sort_values(RANK_SCORE_COL, ascending=False).reset_index(drop=True)

G = int(rank_tbl.shape[0])
rank_tbl["rank"] = np.arange(1, G + 1, dtype=int)
rank_tbl["bin_idx"] = ((rank_tbl["rank"] - 1) // BIN_SIZE + 1).astype(int)
rank_tbl["bin_start"] = ((rank_tbl["bin_idx"] - 1) * BIN_SIZE + 1).astype(int)
rank_tbl["bin_end"] = np.minimum(rank_tbl["bin_idx"] * BIN_SIZE, G).astype(int)
rank_tbl["bin_label"] = rank_tbl["bin_start"].astype(str) + "-" + rank_tbl["bin_end"].astype(str)

# Map bins onto dsn
_dsn = dsn_local.copy()
_dsn["gene"] = _dsn["gene"].astype(str)
_dsn = _dsn.merge(rank_tbl[["gene", "rank", "bin_idx", "bin_label"]], on="gene", how="left")

# ----------------------------
# Build long-form ratio table (target x bin)
# ----------------------------
rows = []
for tgt in TARGETS:
    sub = _dsn[_dsn["target"] == tgt].copy()
    if sub.empty:
        continue

    x = sub["avg_nega"].to_numpy(dtype=float) + EPS_RATIO
    y = sub["avg_posi"].to_numpy(dtype=float) + EPS_RATIO
    ratio = y / x
    log_ratio = np.log2(ratio)

    for g, bidx, blab, r, lr in zip(
        sub["gene"].astype(str).to_numpy(),
        sub["bin_idx"].to_numpy(),
        sub["bin_label"].to_numpy(),
        ratio,
        log_ratio,
    ):
        if pd.isna(bidx) or pd.isna(blab):
            continue
        if np.isfinite(r) and np.isfinite(lr) and r > 0:
            rows.append((tgt, int(bidx), str(blab), g, float(r), float(lr)))

ratio_bin_df = pd.DataFrame(rows, columns=["target", "bin_idx", "bin_label", "gene", "ratio", "log_ratio"])
ratio_bin_df.to_csv(OUT_DIR / "ratio_long_table_bins.csv", index=False)

if ratio_bin_df.empty:
    raise RuntimeError("ratio_bin_df is empty. Check dsn/chil_rank and filters.")

# ----------------------------
# Stats: one-sample Wilcoxon on log2(ratio) vs 0 + BH-FDR
# ----------------------------
stats_rows = []
for tgt in TARGETS:
    for bidx in sorted(ratio_bin_df["bin_idx"].unique()):
        vals = ratio_bin_df[(ratio_bin_df["target"] == tgt) & (ratio_bin_df["bin_idx"] == bidx)]["log_ratio"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        n = int(vals.size)
        if n == 0:
            stats_rows.append({"target": tgt, "bin_idx": bidx, "bin_label": np.nan, "n": 0, "median_ratio": np.nan, "pvalue": np.nan})
            continue

        med_log = float(np.median(vals))
        med_ratio = float(2 ** med_log)
        try:
            _, p = wilcoxon(vals, alternative="two-sided", zero_method="wilcox")
            p = float(p)
        except ValueError:
            p = 1.0

        blab = ratio_bin_df.loc[ratio_bin_df["bin_idx"] == bidx, "bin_label"].iloc[0]
        stats_rows.append({"target": tgt, "bin_idx": bidx, "bin_label": blab, "n": n, "median_ratio": med_ratio, "pvalue": p})

stats_bin_df = pd.DataFrame(stats_rows)
stats_bin_df["qvalue_bh"] = bh_fdr(stats_bin_df["pvalue"].to_numpy(dtype=float))
stats_bin_df.to_csv(OUT_DIR / "stats_wilcoxon_bh_bins.csv", index=False)

# ----------------------------
# Plot: violin + box, grouped by target
# ----------------------------
plt.rcParams.update({
    "font.family": FONT_FAMILY,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

if MAX_RANK_TO_PLOT is None:
    bins_to_plot = sorted(rank_tbl["bin_idx"].unique().tolist())
else:
    bins_to_plot = sorted(rank_tbl.loc[rank_tbl["rank"] <= int(MAX_RANK_TO_PLOT), "bin_idx"].unique().tolist())

plot_df = ratio_bin_df[ratio_bin_df["bin_idx"].isin(bins_to_plot)].copy()
plot_stats = stats_bin_df[stats_bin_df["bin_idx"].isin(bins_to_plot)].copy()

if plot_df.empty:
    raise RuntimeError("No data to plot. Check MAX_RANK_TO_PLOT or input data.")

cmap = plt.cm.get_cmap("viridis", len(bins_to_plot))
colors = [cmap(i) for i in range(len(bins_to_plot))]

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
base_x = np.arange(len(TARGETS), dtype=float) * TARGET_GAP
offsets = np.linspace(-GROUP_SPAN/2, GROUP_SPAN/2, len(bins_to_plot))

all_ratios = plot_df["ratio"].to_numpy(dtype=float)
all_ratios = all_ratios[np.isfinite(all_ratios) & (all_ratios > 0)]
if all_ratios.size == 0:
    raise RuntimeError("No valid ratio values (<=0 or NaN).")

data_min = float(np.min(all_ratios))
data_max = float(np.max(all_ratios))
y_low = data_min / 1.15
y_high = data_max * STAR_Y_MULT * 1.25

if Y_SCALE == "log":
    ax.set_yscale("log", base=2)
    y_low = max(y_low, 1e-6)

ax.set_ylim(y_low, y_high)
ax.axhline(1.0, linestyle="--", linewidth=1)

for i, tgt in enumerate(TARGETS):
    for j, bidx in enumerate(bins_to_plot):
        pos = base_x[i] + offsets[j]
        vals = plot_df[(plot_df["target"] == tgt) & (plot_df["bin_idx"] == bidx)]["ratio"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals) & (vals > 0)]
        if vals.size == 0:
            continue

        v = ax.violinplot([vals], positions=[pos], widths=VIOLIN_WIDTH,
                          showmeans=False, showmedians=False, showextrema=False)
        for body in v["bodies"]:
            body.set_facecolor(colors[j])
            body.set_edgecolor("black")
            body.set_linewidth(0.6)
            body.set_alpha(0.85)

        bp = ax.boxplot([vals], positions=[pos], widths=BOX_WIDTH, showfliers=False,
                        patch_artist=True, manage_ticks=False)
        for box in bp["boxes"]:
            box.set_facecolor("white")
            box.set_edgecolor("black")
            box.set_linewidth(0.8)
        for median in bp["medians"]:
            median.set_color("black")
            median.set_linewidth(1.0)
        for whisker in bp["whiskers"]:
            whisker.set_color("black")
            whisker.set_linewidth(0.8)
        for cap in bp["caps"]:
            cap.set_color("black")
            cap.set_linewidth(0.8)

        if SHOW_POINTS:
            rng = np.random.default_rng(0)
            xj = rng.normal(loc=pos, scale=0.02, size=vals.size)
            ax.scatter(xj, vals, s=POINT_SIZE, alpha=POINT_ALPHA)

        row = plot_stats[(plot_stats["target"] == tgt) & (plot_stats["bin_idx"] == bidx)]
        if not row.empty:
            q = float(row["qvalue_bh"].iloc[0]) if np.isfinite(row["qvalue_bh"].iloc[0]) else np.nan
            star = stars_from_q(q)
            if (star != "NS") or SHOW_NS:
                y_star = float(np.max(vals)) * STAR_Y_MULT
                y_star = min(y_star, y_high * 0.98)
                ax.text(pos, y_star, star, ha="center", va="bottom", fontsize=STAR_FONTSIZE)

ax.set_xticks(base_x)
ax.set_xticklabels(TARGETS, fontsize=TICK_FONTSIZE)
ax.set_xlabel("Histone modification", fontsize=LABEL_FONTSIZE)
ax.set_ylabel("Active / Inactive (CPT ratio)", fontsize=LABEL_FONTSIZE)
ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)

for k in range(len(TARGETS) - 1):
    ax.axvline((base_x[k] + base_x[k+1]) / 2, color="gray", linewidth=0.6, alpha=0.4)

bin_label_map = (rank_tbl.drop_duplicates("bin_idx")[["bin_idx", "bin_label"]]
                 .set_index("bin_idx")["bin_label"].to_dict())
legend_handles = [
    Patch(facecolor=colors[j], edgecolor="black", label=f"Rank {bin_label_map.get(bidx, bidx)}")
    for j, bidx in enumerate(bins_to_plot)
]

if LEGEND_OUTSIDE_RIGHT:
    fig.subplots_adjust(right=LEGEND_RIGHT_MARGIN)
    ax.legend(handles=legend_handles, fontsize=LEGEND_FONTSIZE, frameon=False,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), borderaxespad=0.0)
else:
    ax.legend(handles=legend_handles, fontsize=LEGEND_FONTSIZE, frameon=False, loc="upper right")

fig.tight_layout()
fig.savefig(OUT_PDF, bbox_inches="tight")
plt.show()

print("Saved:", OUT_PDF)
print("Tables saved to:", OUT_DIR)
display(stats_bin_df)

In [ ]:
from matplotlib.patches import Patch

# Split u5d5_n by target (exclude Brg1), then bin by pos_ratio within each target
# and visualize nCell_posi distributions (violin + boxplot).

# ---- Settings ----
BIN_SIZE = int(globals().get("BIN_SIZE_POSRATIO", 500))
EXCLUDE_TARGETS = {"Brg1"}
Y_SCALE = "linear"  # "linear" or "log" (log uses +1 offset)
FIG_W, FIG_H = 9.5, 6.5

TARGET_GAP = 1.0
GROUP_SPAN = 0.72
VIOLIN_WIDTH = 0.11
BOX_WIDTH = 0.06

FONT_FAMILY = "Arial"
LABEL_FONTSIZE = 16
TICK_FONTSIZE = 13
LEGEND_FONTSIZE = 12

OUT_DIR = GAO2026_OUTPUT_ROOT / "05_sci_mtChIL_seq/plot" / f"nCell_posi_violin_box_u5d5n_posratio_bins{BIN_SIZE}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PDF = OUT_DIR / f"nCell_posi_violin_box_u5d5n_posratio_bins{BIN_SIZE}.pdf"
OUT_CSV_LONG = OUT_DIR / "nCell_posi_long_table.csv"
OUT_CSV_SUMMARY = OUT_DIR / "nCell_posi_summary_by_target_bin.csv"

# ---- Safety checks ----
if "u5d5_n" not in globals():
    raise NameError("u5d5_n is not defined. Run the normalization cell first.")

req_cols = {"target", "pos_ratio", "nCell_posi"}
missing = req_cols - set(u5d5_n.columns)
if missing:
    raise KeyError(f"u5d5_n is missing columns: {sorted(missing)}")

# ---- Prepare + binning per target ----
df = u5d5_n.copy()
df["target"] = df["target"].astype(str)
df = df[~df["target"].isin(EXCLUDE_TARGETS)].copy()

df["pos_ratio"] = pd.to_numeric(df["pos_ratio"], errors="coerce")
df["nCell_posi"] = pd.to_numeric(df["nCell_posi"], errors="coerce")

df = df.dropna(subset=["target", "pos_ratio", "nCell_posi"]).copy()

rows = []
for tgt, g in df.groupby("target", sort=False):
    g = g.sort_values("pos_ratio", ascending=False).reset_index(drop=True)
    n = int(len(g))
    if n == 0:
        continue

    rank = np.arange(1, n + 1, dtype=int)
    bin_idx = ((rank - 1) // BIN_SIZE + 1).astype(int)
    bin_start = (bin_idx - 1) * BIN_SIZE + 1
    bin_end = np.minimum(bin_idx * BIN_SIZE, n)
    bin_label = [f"{s}-{e}" for s, e in zip(bin_start, bin_end)]

    gg = g[["target", "pos_ratio", "nCell_posi"]].copy()
    gg["rank"] = rank
    gg["bin_idx"] = bin_idx
    gg["bin_label"] = bin_label
    rows.append(gg)

long_df = pd.concat(rows, axis=0, ignore_index=True)
if long_df.empty:
    raise RuntimeError("No data left after filtering (exclude targets / NA removal).")

# ---- Save tables ----
long_df.to_csv(OUT_CSV_LONG, index=False)

summary_df = (
    long_df.groupby(["target", "bin_idx", "bin_label"], as_index=False)["nCell_posi"]
    .agg(n="size", mean="mean", median="median", q25=lambda s: float(s.quantile(0.25)), q75=lambda s: float(s.quantile(0.75)))
    .sort_values(["target", "bin_idx"])
    .reset_index(drop=True)
 )
summary_df.to_csv(OUT_CSV_SUMMARY, index=False)

# ---- Plot (grouped by target, bins as subgroups) ----
plt.rcParams.update({
    "font.family": FONT_FAMILY,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

TARGET_ORDER = ["H3K27ac", "H3K4me3", "H3K27me3"]
present = set(long_df["target"].astype(str).unique().tolist())
targets = [t for t in TARGET_ORDER if t in present]
bins_to_plot = sorted(long_df["bin_idx"].unique().tolist())

cmap = plt.cm.get_cmap("viridis", len(bins_to_plot))
colors = [cmap(i) for i in range(len(bins_to_plot))]

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

base_x = np.arange(len(targets), dtype=float) * TARGET_GAP
offsets = np.linspace(-GROUP_SPAN / 2, GROUP_SPAN / 2, len(bins_to_plot))

vals_all = long_df["nCell_posi"].to_numpy(dtype=float)
vals_all = vals_all[np.isfinite(vals_all)]
if vals_all.size == 0:
    raise RuntimeError("No valid nCell_posi values.")

if Y_SCALE == "log":
    plot_vals_all = vals_all + 1.0
    ax.set_yscale("log")
    y_min = max(0.8, float(np.min(plot_vals_all)) / 1.2)
    y_max = float(np.max(plot_vals_all)) * 1.25
    ax.set_ylim(y_min, y_max)
else:
    y_min = float(np.min(vals_all)) - 0.05 * (float(np.max(vals_all)) - float(np.min(vals_all)) + 1.0)
    y_max = float(np.max(vals_all)) * 1.05
    ax.set_ylim(y_min, y_max)

for i, tgt in enumerate(targets):
    for j, bidx in enumerate(bins_to_plot):
        pos = base_x[i] + offsets[j]
        sub = long_df[(long_df["target"] == tgt) & (long_df["bin_idx"] == bidx)]
        v = sub["nCell_posi"].to_numpy(dtype=float)
        v = v[np.isfinite(v)]
        if v.size == 0:
            continue

        v_plot = v + 1.0 if Y_SCALE == "log" else v

        vp = ax.violinplot([v_plot], positions=[pos], widths=VIOLIN_WIDTH,
                           showmeans=False, showmedians=False, showextrema=False)
        for body in vp["bodies"]:
            body.set_facecolor(colors[j])
            body.set_edgecolor("black")
            body.set_linewidth(0.6)
            body.set_alpha(0.85)

        bp = ax.boxplot([v_plot], positions=[pos], widths=BOX_WIDTH, showfliers=False,
                        patch_artist=True, manage_ticks=False)
        for box in bp["boxes"]:
            box.set_facecolor("white")
            box.set_edgecolor("black")
            box.set_linewidth(0.8)
        for median in bp["medians"]:
            median.set_color("black")
            median.set_linewidth(1.0)
        for whisker in bp["whiskers"]:
            whisker.set_color("black")
            whisker.set_linewidth(0.8)
        for cap in bp["caps"]:
            cap.set_color("black")
            cap.set_linewidth(0.8)

ax.set_xticks(base_x)
ax.set_xticklabels(targets, fontsize=TICK_FONTSIZE)
ax.set_xlabel("Target", fontsize=LABEL_FONTSIZE)

if Y_SCALE == "log":
    ax.set_ylabel("nCell_posi + 1 (log scale)", fontsize=LABEL_FONTSIZE)
else:
    ax.set_ylabel("Number of active cells per gene", fontsize=LABEL_FONTSIZE)

ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)

for k in range(len(targets) - 1):
    ax.axvline((base_x[k] + base_x[k + 1]) / 2, color="gray", linewidth=0.6, alpha=0.4)

legend_handles = [
    Patch(facecolor=colors[j], edgecolor="black", label=f"Rank {((bidx-1)*BIN_SIZE+1)}-{bidx*BIN_SIZE}")
    for j, bidx in enumerate(bins_to_plot)
]
fig.subplots_adjust(right=0.78)
ax.legend(handles=legend_handles, fontsize=LEGEND_FONTSIZE, frameon=False,
          loc="upper left", bbox_to_anchor=(1.01, 1.0), borderaxespad=0.0)

ax.set_title("")

fig.tight_layout()
fig.savefig(OUT_PDF, bbox_inches="tight")
plt.show()

print("Saved:", OUT_PDF)
print("Saved:", OUT_CSV_LONG)
print("Saved:", OUT_CSV_SUMMARY)
display(summary_df)